# Reviewer-response analyses

Runs the three analyses added during revision and fills in the gradient-boosting
rows that need XGBoost:

1. `prevalence_sensitivity.py` — precision and alert volume at 4%, 1% and 0.1%
2. `reviewer_response.py` — paired bootstrap intervals on the attack-focused
   metrics, plus the imbalance-aware FFNN baseline
3. `imbalance_baselines.py` — class weighting vs SMOTE vs under-sampling vs a
   tuned threshold, on the same data and the same split

Runtime → Change runtime type → **CPU**. No GPU needed.
Total time is roughly 20–40 minutes, most of it in step 3.

## 1. Clone and install

Colab ships numpy, pandas, scikit-learn and matplotlib already, and its own
`google-colab` package pins some of them. Installing this repository's pinned
`requirements.txt` on top of that upgrades numpy underneath packages that were
compiled against the older one, and the session dies with
`cannot import name '_slice' from 'numpy._core.umath'`.

So on Colab, install only what is genuinely missing and leave the rest alone.
Small differences in the third decimal are possible against the published
numbers; that is an acceptable trade for a working runtime. Use the pinned file
only in a local virtual environment, where nothing else depends on those
versions.

In [ ]:
!git clone https://github.com/moh1984/ids-metadata-evaluation-protocol.git
%cd ids-metadata-evaluation-protocol

# Only the two packages Colab lacks. Do NOT install requirements.txt here.
!pip install -q xgboost shap

import numpy, pandas, sklearn, matplotlib, xgboost, shap
print("numpy       ", numpy.__version__)
print("pandas      ", pandas.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib  ", matplotlib.__version__)
print("xgboost     ", xgboost.__version__, "  <- gradient boosting rows need this")
print("shap        ", shap.__version__)

### Do the revision scripts exist in the clone?

The three scripts below were written after the repository was first published,
so a clone only contains them once the updated repository has been pushed. The
next cell checks, and if any are missing it offers an upload box: pick
`prevalence_sensitivity.py`, `reviewer_response.py` and `imbalance_baselines.py`
from your computer and they will be placed in `src/`.

Pushing the updated repository to GitHub is the better fix, since it also keeps
the public record consistent with the paper. The upload box is here so you are
not blocked in the meantime.

In [ ]:
import os, shutil

NEEDED = ["prevalence_sensitivity.py", "reviewer_response.py",
          "imbalance_baselines.py", "repeated_splits.py"]
missing = [f for f in NEEDED if not os.path.exists(f"src/{f}")]

if not missing:
    print("all three revision scripts are present in the clone")
else:
    print("missing from the clone:", ", ".join(missing))
    print("upload them now (you can select all three at once):")
    from google.colab import files
    up = files.upload()
    for name in up:
        shutil.move(name, f"src/{name}")
        print("  placed src/" + name)
    still = [f for f in NEEDED if not os.path.exists(f"src/{f}")]
    print("still missing:", still if still else "nothing - ready to run")

## 2. Prevalence sensitivity (seconds)

Pure arithmetic from the held-out TPR and FPR already in Table 4, so this needs
no training and no XGBoost. It produces Table 6 and Fig. 5.

In [ ]:
!python src/prevalence_sensitivity.py

## 3. Bootstrap intervals and the balanced FFNN (about 10 minutes)

Refits the five tuned configurations, then resamples the held-out predictions
2,000 times with the same index applied to every model, so the comparisons are
paired. With XGBoost present the gradient-boosting rows appear, which is what
was missing from the earlier run.

This now also writes `table12b_paired_differences.csv`, the paired
difference intervals that the significance claims rest on, and reports the
feed-forward network under sample weighting as well as under resampling.

In [ ]:
!python src/reviewer_response.py --boot 2000

## 4. Matched imbalance-strategy comparison (about 20–30 minutes)

Holds the data, the split and the model configurations fixed and varies only the
imbalance strategy. Strategy D refits five cross-validation folds per model to
choose the threshold, which is where most of the time goes.

In [ ]:
!python src/imbalance_baselines.py

## 5. Repeated splits (about 15–25 minutes)

Reviewer 2 asked whether the model ordering survives resampling of the split.
This repeats the whole binary experiment over five stratified splits, with the
hyperparameters fixed at the Table 1 values so that only the split changes. It
reports the mean and spread of each metric and how often each model takes first
place on PR-AUC.

In [ ]:
!python src/repeated_splits.py --seeds 42 1 7 13 2024

## 6. Read the results

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

for name in ["table11_prevalence_sensitivity",
             "table12_bootstrap_intervals",
             "table12b_paired_differences",
             "table13_ffnn_balanced",
             "table14_imbalance_strategies",
             "table15_repeated_splits"]:
    display(Markdown("### `" + name + "`"))
    display(pd.read_csv("results/tables/" + name + ".csv").round(4))

## 7. Download the new tables

In [ ]:
from google.colab import files

!zip -qr revision_tables.zip results/tables/table1*.csv results/tables/mc_best_params.json results/figures/fig10*.png
files.download("revision_tables.zip")

---
### If you would rather run it locally

```bash
git clone https://github.com/moh1984/ids-metadata-evaluation-protocol.git
cd ids-metadata-evaluation-protocol
python -m venv .venv && source .venv/bin/activate     # Windows: .venv\Scripts\activate
pip install -r requirements.txt
python src/prevalence_sensitivity.py
python src/reviewer_response.py --boot 2000
python src/imbalance_baselines.py
```

A virtual environment matters here: `requirements.txt` pins exact versions, and
installing them system-wide can break other projects. If `pip install xgboost`
fails on Windows, install the Microsoft Visual C++ Redistributable first.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: xgboost` | `!pip install -q xgboost`, then re-run the cell |
| `can't open file '.../src/prevalence_sensitivity.py'` | The clone predates the revision scripts. Push the updated repository, or use the upload cell in step 1 |
| Gradient-boosting rows missing from a table | XGBoost was not importable when that script ran |
| Numbers differ in the third decimal | A different XGBoost or scikit-learn build. Expected on Colab; use a local virtual environment with the pinned `requirements.txt` for an exact match |
| Session disconnects during step 4 | Re-run it; steps 2 and 3 are already saved to `results/tables/` |
| `cannot import name '_slice' from 'numpy._core.umath'` | numpy was upgraded under a running session. Runtime → Restart session, then re-run from cell 1 without installing `requirements.txt` |
| `google-colab 1.0.0 requires pandas==2.2.3` | Harmless if you only installed xgboost and shap; fatal if you installed the pinned file. Restart the session |